# Named Entity Recognition (NER)

In [2]:
import spacy

# Deutsches Modell laden
nlp = spacy.load("de_core_news_sm")

text = "Angela Merkel flog mit der Lufthansa von Berlin nach New York."
doc = nlp(text)

print(f"{'Entität':<20} | {'Label':<10} | {'Erklärung'}")
print("-" * 50)

for ent in doc.ents:
    # ent.label_ gibt den Kategorienamen zurück (z.B. PER, ORG, LOC)
    print(f"{ent.text:<20} | {ent.label_:<10} | {spacy.explain(ent.label_)}")


Entität              | Label      | Erklärung
--------------------------------------------------
Angela Merkel        | PER        | Named person or family.
Lufthansa            | ORG        | Companies, agencies, institutions, etc.
Berlin               | LOC        | Non-GPE locations, mountain ranges, bodies of water
New York             | LOC        | Non-GPE locations, mountain ranges, bodies of water


# Dependency Parsing → Syntax & Relationen
spaCy analysiert wer was mit wem macht:
Subjekt, Objekt, Prädikat
➡️ Grundlage für:

juristische Analyse

Question Answering

Regelbasierte Systeme

Für dein juristisches NLP-Interesse ist das extrem relevant.


In [6]:
import spacy

nlp = spacy.load("de_core_news_sm")

text = "Die Katzen liefen schneller als der Hund."
text1 = "Angela Merkel flog mit der Lufthansa von Berlin nach New York."
doc = nlp(text1)

print(f"{'Text':<10} | {'Dep':<10} | {'Head':<10} | {'Erklärung'}")
print("-" * 55)

for token in doc:
    print(f"{token.text:<10} | {token.dep_:<10} | {token.head.text:<10} | {spacy.explain(token.dep_)}")


Text       | Dep        | Head       | Erklärung
-------------------------------------------------------
Angela     | pnc        | Merkel     | proper noun component
Merkel     | sb         | flog       | subject
flog       | ROOT       | flog       | root
mit        | mo         | flog       | modifier
der        | nk         | Lufthansa  | noun kernel element
Lufthansa  | nk         | mit        | noun kernel element
von        | mnr        | Lufthansa  | postnominal modifier
Berlin     | nk         | von        | noun kernel element
nach       | mnr        | Lufthansa  | postnominal modifier
New        | pnc        | York       | proper noun component
York       | nk         | nach       | noun kernel element
.          | punct      | flog       | punctuation


In [7]:

from spacy import displacy
from IPython.display import display, HTML  # Korrekter Import für neuere IPython-Versionen

# 1. HTML-String generieren (jupyter=False)
html = displacy.render(doc, style="dep", jupyter=False)

# 2. Manuell in Jupyter anzeigen
display(HTML(html))


# Textklassifikation & Sentiment (nicht nur Tokenizing!)
Vorbereitung:

pip install spacy-sentiws und SentiWS-Daten herunterladen.

SentiWS_v2.0_Positive.txt
https://wortschatz.uni-leipzig.de/de/download/


In [2]:
import os
import spacy
from spacy_sentiws import spaCySentiWS

senti_path = r"C:\0_DA\Python_practice\nltk_spacy\data"

# Test: Welche Dateien sieht Python in dem Ordner?
print("Dateien im Ordner:", os.listdir(senti_path))

nlp = spacy.load("de_core_news_sm")

# Falls die Komponente schon drin ist, erst entfernen für sauberen Neustart
if "sentiws" in nlp.pipe_names:
    nlp.remove_pipe("sentiws")

# Neu hinzufügen
nlp.add_pipe('sentiws', config={'sentiws_path': senti_path})

# Test mit Lemmatisierung (SentiWS nutzt oft Grundformen)
doc = nlp("Das ist gut .")
for token in doc:
    print(f"Token: {token.text}, Lemma: {token.lemma_}, Sentiment: {token._.sentiws}")

doc = nlp("Das Wetter ist heute hervorragend, aber die Bahnverspätung ist schrecklich.")
for token in doc:
    print(f"Token: {token.text}, Lemma: {token.lemma_}, Sentiment: {token._.sentiws}")



Dateien im Ordner: ['SentiWS_v2.0_Negative.txt', 'SentiWS_v2.0_Positive.txt']
Token: Das, Lemma: der, Sentiment: None
Token: ist, Lemma: sein, Sentiment: None
Token: gut, Lemma: gut, Sentiment: None
Token: ., Lemma: --, Sentiment: None
Token: Das, Lemma: der, Sentiment: None
Token: Wetter, Lemma: Wetter, Sentiment: None
Token: ist, Lemma: sein, Sentiment: None
Token: heute, Lemma: heute, Sentiment: None
Token: hervorragend, Lemma: hervorragend, Sentiment: None
Token: ,, Lemma: --, Sentiment: None
Token: aber, Lemma: aber, Sentiment: None
Token: die, Lemma: der, Sentiment: None
Token: Bahnverspätung, Lemma: Bahnverspätung, Sentiment: None
Token: ist, Lemma: sein, Sentiment: None
Token: schrecklich, Lemma: schrecklich, Sentiment: None
Token: ., Lemma: --, Sentiment: None


# Manueller SentiWS-Abgleich

In [16]:
import spacy

nlp = spacy.load("de_core_news_sm")

# 1. SentiWS Dateien manuell in ein Dictionary laden
senti_dict = {}
senti_files = [
    r"C:\0_DA\Python_practice\nltk_spacy\data\SentiWS_v2.0_Positive.txt",
    r"C:\0_DA\Python_practice\nltk_spacy\data\SentiWS_v2.0_Negative.txt"
]

for file_path in senti_files:
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.split("\t")
            # Format: Lemma|Wortart  Sentimentwert  Flexionen
            lemma_info = parts[0].split("|")[0]
            value = float(parts[1])
            senti_dict[lemma_info] = value

# 2. Test-Satz verarbeiten
doc = nlp("Das Wetter ist heute hervorragend, aber die Bahnverspätung ist schrecklich.")

# 3. Abgleich über das Lemma
print(f"{'Wort':<15} | {'Lemma':<15} | {'Sentiment':<10}")
print("-" * 45)

for token in doc:
    # Suche das Lemma im Dictionary
    sentiment = senti_dict.get(token.lemma_, 0.0)
    
    if sentiment != 0.0:
        print(f"{token.text:<15} | {token.lemma_:<15} | {sentiment:>10}")


Wort            | Lemma           | Sentiment 
---------------------------------------------
hervorragend    | hervorragend    |     0.5891
schrecklich     | schrecklich     |    -0.0242


In [23]:
#\t = Tabulatoren
line="Hallo\t\tguten\tTag"
line="Abmachung|NN	0.0040	Abmachungen"

parts = line.split("\t")
print(parts)
parts = line.split(" ")
print(parts)

['Abmachung|NN', '0.0040', 'Abmachungen']
['Abmachung|NN\t0.0040\tAbmachungen']
